# Our RSGM against the pretrained PoseD-Flow, on their data (19 joints)


A quick comparison: the prior of PoseD-Flow (Nadar, Foti, Birdal, CVPR 2026), their released checkpoint, against our RSGM trained on **their** AMASS (the preprocessing of DPoser, converted by `scripts/preprocess/convert_posedflow.py`, split `posedflow`).

- **Joints.** Their model is on SO(3)^21 (SMPL joints 1–21), ours on 19 (without the toes). Their samples are cut to our 19 joints (`hp/models/posedflow.py`). The toes are leaves: they move no other joint.
- **Data.** Train: every 10th of their 18.6M training poses (1.86M). Test: all 54,200 test poses. Their checkpoint saw the whole train set; ours the 10% subset (neighbouring frames at 30 Hz are nearly the same).
- **Metrics.**
  1. our harness (`hp.evaluation.evaluate`): Wasserstein, recall, precision against the test poses, with the training poses as the yardstick;
  2. the protocol of their paper (Tab. 1; `hp/evaluation/posedflow_protocol.py`), on 20 sets × 500 samples, mean ± 95% CI:
     - **FID**: Fréchet distance of the 22 SMPL joint positions (SMPL neutral v1.1.0, betas = 0) against their precomputed statistics `gt_mean_var_v2.npz`. Also against the statistics of our train subset with the same skeleton ("FID own"), which is self-consistent: their statistics sit ~0.005 away in the mean, likely another SMPL file;
     - **APD**: mean pairwise joint distance, cm;
     - **dNN**: distance to the nearest training pose. Theirs is on 21 joints against all 18.6M training poses. Here it's on 19 joints against the 1.86M of the subset, so it's larger than in their table, for both models alike.

  Their checkpoint reproduces their table here: FID 0.014 and APD 15.5 cm, against 0.013–0.014 and 15.4–15.5 in the paper.
- **Likelihood.** It is not comparable. A CNF density on SO(3)^21 cannot be marginalized to 19 joints, so their model has no log-likelihood on our joints. RSGM's likelihood is printed for the record.
- **Their sampler.** The paper uses midpoint with 100 steps and their script Euler with 1000. Both are evaluated. Their script also samples with the raw weights, not the EMA ones, because it never calls `eval()`. dNN differs by 0.0016 rad between the two (a quick check).

In [ ]:
import json
import re
import sys
from pathlib import Path

# the code of the checkout this notebook is in (e.g. a worktree), before an editable install of another one
SRC = next(str(p / "src") for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "hp").is_dir())
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import numpy as np
import torch

from hp.evaluation.data import load_reference
from hp.evaluation.evaluate import EvalConfig, evaluate
from hp.evaluation.posedflow_protocol import SmplSkeleton, apd, confidence, fid, nearest_neighbour_distance
from hp.models.io import MANIFEST, load_model
from hp.models.posedflow import PoseDFlowPretrained

assert load_model.__module__ == "hp.models.io" and sys.modules["hp.models.io"].__file__.startswith(SRC), \
    f"hp comes from {sys.modules['hp.models.io'].__file__}: restart the kernel"

# ---- parameters ----------------------------------------------------------------------
ROTATIONS = "~/data/AMASS_rotations_posedflow_c10"  # convert_posedflow.py --train-stride 10 --valid-stride 100
CHECKPOINT = "~/data/posedflow_ckpt/checkpoints/epoch-009_step-0_loss-0.0000.ckpt"
SKELETON = "~/data/body_models/smpl/smpl_neutral_skeleton.npz"  # scripts/preprocess/convert_smpl_skeleton.py
GT_STATS = "~/Documents/PoseDFlow/examples/gt_mean_var_v2.npz"  # their FID statistics (muj, covj)

# Any number of models; comment a line out to leave a model out. Two kinds:
#   posedflow=dict(weights=, method=, sample_steps=)   their checkpoint (hp/models/posedflow.py)
#   run=<run folder, or a folder of runs: the newest finished one>, optional weights= (default weights_ema),
#       sample_steps=, likelihood_steps= (set on the loaded model; default: as trained)
MODELS = {
    "PoseD-Flow (midpoint 100)": dict(posedflow=dict(weights="ema", method="midpoint", sample_steps=100)),
    "PoseD-Flow (Euler 1000, their script)": dict(posedflow=dict(weights="raw", method="euler", sample_steps=1000)),
    "RSGM t23, our AMASS": dict(run="~/data/experiments/rsgm_tuned_t23"),
    "RSGM 1536x6, their data": dict(run="~/data/experiments/rsgm_posedflow_s10"),
    "RSGM 512x4, their data": dict(run="~/data/experiments/rsgm_posedflow_small_s10"),
    # "RSGM 1536x6, their data, 1000 steps": dict(run="~/data/experiments/rsgm_posedflow_s10", sample_steps=1000),
}

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
HARNESS = EvalConfig()  # 20k samples, 2000 queries, 3 repeats
N_CONTRAST = 100_000  # training poses as the yardstick of the harness
SETS, SET_SIZE = 20, 500  # as their paper
CACHE = Path("~/data/experiments/posedflow_comparison/cache").expanduser()  # cache/<model>/<stage>.json
RECOMPUTE = set()  # names of models to evaluate again although their cache is valid; True: all
# --------------------------------------------------------------------------------------
CACHE.mkdir(parents=True, exist_ok=True)


def run_folder(path):
    path = Path(path).expanduser()
    if (path / MANIFEST).is_file():
        return path
    runs = sorted(p for p in path.glob("run_[0-9][0-9][0-9][0-9]") if (p / "history.json").is_file() and (p / MANIFEST).is_file())
    return runs[-1] if runs else None

## 1. The data

In [ ]:
test = load_reference("test", ROTATIONS, "posedflow")
train = load_reference("train", ROTATIONS, "posedflow")
contrast = train[torch.from_numpy(np.random.default_rng(0).choice(len(train), N_CONTRAST, replace=False))]
print(f"test {len(test):,}   train {len(train):,}   contrast {len(contrast):,}")

## 2. The models

Every entry of `MODELS` is built from its spec:
- **PoseD-Flow**: their checkpoint, trained and evaluated on their data.
- **RSGM runs**, which may come from different data:
  - *their data*: trained on their train set (`explore/training/rsgm_posedflow_train.ipynb`). This is the fair comparison;
  - *our AMASS*: trained on our AMASS, the SMPL-X G fit at all frames. Theirs is a SMPL fit at 30 Hz with the first and last 10% of every sequence cut. The datasets of the split are the same (the VPoser split: test Transitions + SSM). Such a model is evaluated on data it has never seen, so the comparison favours PoseD-Flow.

**The cache.** Every model has its own folder `cache/<name>/` with one file per stage (`harness.json`, `paper.json`). A file also stores what it was computed from: the spec, the run folder, the size and time of the weights file, and the settings of the evaluation. A saved result is reused only if all of these match, so a retrained run or a changed setting is evaluated again by itself. Renaming a model starts a new folder.

In [ ]:
def resolve(name, spec):
    """(builder, identity): how to build the model, and what its results depend on (for the cache)."""
    if "posedflow" in spec:
        args = dict(spec["posedflow"])
        checkpoint = Path(CHECKPOINT).expanduser()
        identity = {"kind": "posedflow", "args": args, "checkpoint": str(checkpoint), "mtime": checkpoint.stat().st_mtime}
        return (lambda: PoseDFlowPretrained(checkpoint, **args).to(DEVICE)), identity
    run = run_folder(spec["run"])
    if run is None:
        return None, None
    weights = spec.get("weights", "weights_ema")
    file = run / f"{weights}.pt"
    overrides = {k: spec[k] for k in ("sample_steps", "likelihood_steps") if k in spec}
    identity = {"kind": "run", "run": str(run), "weights": weights, "size": file.stat().st_size,
                "mtime": file.stat().st_mtime, "overrides": overrides}

    def build():
        model = load_model(run, device=DEVICE, weights=weights)
        for k, v in overrides.items():
            setattr(model, k, v)
        return model
    return build, identity


builders, identities = {}, {}
for name, spec in MODELS.items():
    build, identity = resolve(name, spec)
    if build is None:
        print(f"{name}: skipped, no finished run in {spec['run']}")
        continue
    builders[name], identities[name] = build, identity
    print(f"{name}: {identity.get('run', identity.get('checkpoint'))}")


def slug(name):
    return re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")


def cached(name, stage, settings, compute):
    """The result of `stage` for model `name`, from cache/<name>/<stage>.json if it was computed with these settings."""
    path = CACHE / slug(name) / f"{stage}.json"
    settings = json.loads(json.dumps(settings))  # as it reads back from json
    if path.is_file() and not (RECOMPUTE is True or name in RECOMPUTE):
        saved = json.loads(path.read_text())
        if saved.get("settings") == settings:
            return saved["result"]
        print(f"  {name} / {stage}: the cached result is of other settings, evaluated again")
    result = compute()
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps({"name": name, "stage": stage, "settings": settings, "result": result}, indent=1))
    return result

## 3. Our harness: Wasserstein, recall, precision

In [ ]:
harness_settings = {"config": HARNESS.__dict__, "rotations": ROTATIONS, "test": len(test), "contrast": N_CONTRAST}
harness = {}
for name, build in builders.items():
    harness[name] = cached(name, "harness", {"model": identities[name], **harness_settings},
                           lambda build=build: evaluate(build(), test, HARNESS, contrast=contrast, progress=True))
    print(f"--- {name} done")

In [ ]:
def pm(d, key="mean_deg"):
    return f"{d[key]['mean']:6.2f} ± {d[key]['std']:4.2f}"


print(f"{'':45s} {'Wasserstein':>14s} {'recall':>14s} {'precision':>14s}   log-lik. (space)")
for name, r in harness.items():
    ll = r["likelihood"]
    ll = "—" if ll is None else f"{ll['mean']:.2f} ({ll['space']})"
    print(f"{name:45s} {pm(r['wasserstein']['generated'], 'joint_mean_deg'):>14s} {pm(r['recall']):>14s} {pm(r['precision']):>14s}   {ll}")
r = next(iter(harness.values()))  # the yardstick is the same for every model
print(f"{'real train poses (yardstick)':45s} {pm(r['wasserstein']['contrast'], 'joint_mean_deg'):>14s} "
      f"{pm(r['contrast_recall']):>14s} {pm(r['contrast_precision']):>14s}")
print("degrees per joint (mean over the joints); mean ± std over the repeats; lower is better")

## 4. The protocol of their paper: FID, APD, dNN

The same 20 sets of 500 poses per model for all three metrics. Their Tab. 1 for reference (21 joints, dNN against all their training poses):

| | FID ↓ | APD (cm) ↑ | dNN (rad) ↓ |
|---|---|---|---|
| VPoser | 0.048 ± .002 | 14.684 ± .138 | 0.074 ± .000 |
| DPoser | 0.019 ± .001 | 14.992 ± .123 | 0.073 ± .001 |
| PoseFM | 0.016 ± .001 | 14.762 ± .176 | 0.079 ± .000 |
| PoseRFM (N=100) | 0.014 ± .001 | 15.481 ± .124 | 0.069 ± .001 |
| PoseRFM (N=1000) | 0.013 ± .001 | 15.544 ± .167 | 0.070 ± .001 |

In [ ]:
skeleton = SmplSkeleton(SKELETON)
gt = np.load(Path(GT_STATS).expanduser())
muj, covj = gt["muj"].astype(np.float64), gt["covj"].astype(np.float64)
own = skeleton.joints(train).reshape(len(train), -1).numpy()  # the statistics of our train subset, same skeleton
mu_own, cov_own = own.mean(0), np.cov(own, rowvar=False)
del own
train_dev = train.to(DEVICE)


def protocol(make_set):
    """FID (their statistics and ours), APD and dNN over SETS sets of SET_SIZE poses [SET_SIZE, 19, 3, 3]."""
    out = {"FID": [], "FID own": [], "APD": [], "dNN": []}
    for k in range(SETS):
        poses = make_set(k)
        joints = skeleton.joints(poses)
        out["FID"].append(fid(joints, muj, covj))
        out["FID own"].append(fid(joints, mu_own, cov_own))
        out["APD"].append(apd(joints))
        out["dNN"].append(nearest_neighbour_distance(poses, train_dev).mean().item())
    return {name: confidence(values) for name, values in out.items()}


paper_settings = {"sets": SETS, "size": SET_SIZE, "rotations": ROTATIONS, "train": len(train), "skeleton": SKELETON,
                  "gt_stats": GT_STATS}


def real(poses, seed):
    rng = np.random.default_rng(seed)
    return lambda k: poses[torch.from_numpy(rng.choice(len(poses), SET_SIZE, replace=False))]


def generated(build):
    def compute():
        model = build().eval()
        torch.manual_seed(0)
        return protocol(lambda k: model.sample(SET_SIZE).float().cpu())
    return compute


paper = {
    "real test poses": cached("real test poses", "paper", {"test": len(test), **paper_settings}, lambda: protocol(real(test, 1))),
    "real train poses": cached("real train poses", "paper", paper_settings, lambda: protocol(real(train, 2))),
}
for name, build in builders.items():
    paper[name] = cached(name, "paper", {"model": identities[name], **paper_settings}, generated(build))

print(f"{'':45s} {'FID':>16s} {'FID own':>16s} {'APD (cm)':>16s} {'dNN (rad)':>16s}")
for name, r in paper.items():
    cells = [f"{r[m]['mean']:.4f} ± {r[m]['ci95']:.4f}" if m != "APD" else f"{r[m]['mean']:.2f} ± {r[m]['ci95']:.2f}" for m in ("FID", "FID own", "APD", "dNN")]
    print(f"{name:45s} " + " ".join(f"{c:>16s}" for c in cells))

## 5. Reading

- FID of real test poses is not 0: the test subjects differ from the training ones that the statistics come from, and 500 poses give a biased covariance. The real train poses are what the statistics describe.
- The real train poses are the poses the statistics describe: their FID is the floor of 500-pose sets (about 0.011), and their dNN is 0 by construction (each is its own nearest neighbour).
- dNN alone rewards memorization: a model that returns training poses has dNN 0. Read it with recall and precision. Real unseen test poses give the scale of a good value.
- Recall (real → nearest generated) shows coverage; precision (generated → nearest real) shows realism. The contrast row is what real poses of other subjects score.